# **[_Processing JSON Data into Databricks_](url)**

In [0]:
%sql
-- Use the catalog
USE CATALOG pysaprk_demo;
-- Create a schema
CREATE SCHEMA IF NOT EXISTS json_demo;
-- Use the schema
USE SCHEMA json_demo;


In [0]:
# Get current catalog and schema from SQL session
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema = spark.sql("SELECT current_schema()").collect()[0][0]

# Create volume using dynamic values
spark.sql(f"CREATE VOLUME IF NOT EXISTS {current_catalog}.{current_schema}.json_data")

In [0]:
%sql
SELECT *
FROM json.`/Volumes/pysaprk_demo/json_demo/json_data/kafka_messages_01.json`
limit 2
;

In [0]:
%sql
SELECT 
  *
FROM read_files(
  '/Volumes/pysaprk_demo/json_demo/json_data/kafka_messages_01.json',
  format => 'json',
  inferSchema => true,
  multiline => true
)

## Using CTAS and read_files() with JSON

In [0]:
%sql
-- 1. Drop the table if exists for reproducibility
DROP TABLE IF EXISTS kafka_events_bronze_raw;

-- 2. Create the Delta table
CREATE TABLE IF NOT EXISTS kafka_events_bronze_raw
AS
SELECT 
  *
FROM read_files(
  '/Volumes/pysaprk_demo/json_demo/json_data/kafka_messages_01.json',
  format => 'json',
  inferSchema => true,
  multiline => true
);

-- 3. Display the table
SELECT * FROM kafka_events_bronze_raw limit 2;

## Decoding base64 Strings for the Bronze Table

In [0]:
%sql
select 
  key as encoded_key_value,
  unbase64(key) as decoded_key_value,
  value as encoded_value,
  unbase64(value) as decoded_value
from pysaprk_demo.json_demo.kafka_events_bronze_raw
limit 5
;

In [0]:
%sql
select 
  key as encoded_key_value,
  cast(unbase64(key) AS STRING) as decoded_key_value,
  value as encoded_value,
  cast(unbase64(value) AS STRING) as decoded_value
from pysaprk_demo.json_demo.kafka_events_bronze_raw
limit 5
;

In [0]:
%sql
-- 1. Drop the table if exists for reproducibility
DROP TABLE IF EXISTS tb_kafka_events_bronze_decoded;

-- 2. Create the Delta table
CREATE OR REPLACE TABLE tb_kafka_events_bronze_decoded
AS
SELECT 
  cast(unbase64(key) AS STRING) as decoded_key,
  offset,
  partition,
  timestamp,
  timestampType,
  topic,
  cast(unbase64(value) AS STRING) as decoded_value
FROM pysaprk_demo.json_demo.kafka_events_bronze_raw
;

-- 3. Display the table
SELECT * FROM tb_kafka_events_bronze_decoded limit

## Working with JSON Formatted Strings in a Table

#### Flattening JSON String Columns

In [0]:
%sql
SELECT 
  decoded_value,
  decoded_value:order_id as order_id,
  decoded_value:email as email,
  decoded_value:transaction_timestamp as transaction_timestamp,
  decoded_value:total_item_quantity as total_item_quantity,
  decoded_value:purchase_revenue_in_usd as purchase_revenue_in_usd,
  decoded_value:unique_items as unique_items,
  decoded_value:items as items,
  decoded_value:event_type as event_type,
  decoded_value:currency as currency
FROM tb_kafka_events_bronze_decoded
limit 2
;

In [0]:
%sql
-- 1. Drop the table if exists for reproducibility
DROP TABLE IF EXISTS tb_kafka_events_bronze_string_flattened;

-- 2. Create the Delta table
CREATE OR REPLACE TABLE tb_kafka_events_bronze_string_flattened
AS
SELECT 
  decoded_key,
  offset,
  partition,
  timestamp,
  timestampType,
  topic,
  -- decoded_value,
  decoded_value:order_id as order_id,
  decoded_value:email as email,
  decoded_value:transaction_timestamp as transaction_timestamp,
  decoded_value:total_item_quantity as total_item_quantity,
  decoded_value:purchase_revenue_in_usd as purchase_revenue_in_usd,
  decoded_value:unique_items as unique_items,
  decoded_value:items as items,
  decoded_value:event_type as event_type,
  decoded_value:currency as currency
FROM tb_kafka_events_bronze_decoded
;

-- 3. Display the table
SELECT * FROM tb_kafka_events_bronze_string_flattened limit 2;


## Flattening JSON Formating Strings via STRUCT Conversion

#### Converting a JSON STRING to a STRUCT Column

In [0]:
%sql
SELECT schema_of_json('[{"item_id":"P002","item_name":"Running Shoes","category":"Footwear","unit_price":89.95,"quantity":3,"subtotal":269.85}]') as schema


In [0]:
%sql
CREATE OR REPLACE TABLE tb_kafka_events_bronze_struct
AS
SELECT 
  * EXCEPT(items),
  from_json(
    items,
    'ARRAY<STRUCT<category: STRING, item_id: STRING, item_name: STRING, quantity: BIGINT, subtotal: DOUBLE, unit_price: DOUBLE>>') as items_values
FROM tb_kafka_events_bronze_string_flattened
;
    
SELECT * FROM tb_kafka_events_bronze_struct limit 2;

In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.types import *

In [0]:
%sql
SELECT 
  x.*,
  items.category,
  items.item_id,
  items.item_name,
  items.quantity,
  items.subtotal,
  items.unit_price
FROM (
        SELECT 
        *,
        explode(items_values) as items
      FROM tb_kafka_events_bronze_struct
) x